# Time-evolution EUV movie

Renders a synthetic AIA frame per timestep and assembles a movie with `rrt.viz.make_movie` (ffmpeg → imageio → opencv → GIF fallback). Here the 'evolution' is a synthetic density that brightens with time; swap the loop body for `load_mhd(f'snap_{t}.h5')` on real snapshots.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from rrt import synthdata
from rrt.observer import make_observer, image_grid
from rrt.euv import run_siddon
from rrt.response import FLAT_T, FLAT_R
from rrt.viz import aia_cmap, plot_map, make_movie

In [ ]:
c = synthdata.make_cube('full', 'density_only', nr=24, nth=36, nph=48, r_range=(1.0, 3.0))
r, th, ph, ne0 = c['r'], c['theta'], c['phi'], c['ne']
T = np.full_like(ne0, 1.0e6)
RMAX, NPIX = 1.5, 160
Xg, Yg, _ = image_grid(RMAX, NPIX)
_, e_los, x_img, y_img = make_observer(-70, 0)
os.makedirs('frames_demo', exist_ok=True)

In [ ]:
paths = []
for i, boost in enumerate(np.linspace(1.0, 2.0, 6)):
    em = run_siddon(Xg, Yg, x_img, y_img, e_los, r, th, ph, ne0*boost, T,
                    FLAT_T, FLAT_R, Npix=NPIX, Rmax=RMAX, plot=False, save=False)
    fig, _, _ = plot_map(em, RMAX, cmap=aia_cmap(171), log=True, title=f't={i}')
    p = f'frames_demo/frame_{i:02d}.png'; fig.savefig(p, dpi=110); plt.close(fig)
    paths.append(p)
out = make_movie(paths, 'euv_demo.mp4', fps=4)
print('movie →', out)